# DuckDB recipe store example

Shows an in-memory DuckDB recipe store. Use `DuckDBRecipeStore("recipes.duckdb")` for a persistent file.

In [ ]:
from woodpecker.recipes import RecipeLoader
from woodpecker.stores import DuckDBRecipeStore
from woodpecker.testing import make_atlas, make_cmip6, make_cmip6_decadal, make_cmip7

Load discovered recipes into the DuckDB store.

In [ ]:
store = DuckDBRecipeStore()

for recipe in RecipeLoader().catalog().list_recipes():
    store.save_recipe(recipe)

[recipe.id for recipe in store.list_recipes()]

Query representative datasets by metadata and source path.

In [ ]:
query_cases = [
    (
        "CMIP6 core",
        make_cmip6(overrides={"units": "degC"}, seed=7),
        "CMIP6.CMIP.MOHC.HadGEM3-GC31-LL.historical.r1i1p1f3.Amon.tas.gn.v20200101.nc",
    ),
    (
        "CMIP6 decadal",
        make_cmip6_decadal(seed=7),
        "CMIP6.DCPP.MPI-M.MPI-ESM1-2-HR.dcppA-hindcast.s1960-r1i1p1f1.Omon.tos.gn.v20200101.nc",
    ),
    (
        "Atlas",
        make_atlas(seed=7),
        "c3s-ipcc-atlas.cmip6.historical.ssp245.pr.monthly.global.nc",
    ),
    (
        "ESA CCI water vapour",
        make_cmip7(variable="prw", seed=7),
        "ESACCI-WATERVAPOUR-L3C-TCWV-meris-005deg-2002-2017-fv3.2.zarr",
    ),
]

In [ ]:
matches = []
for label, dataset, source_path in query_cases:
    matched_plans = store.lookup(dataset, path=source_path)
    matches.append(
        {
            "case": label,
            "dataset_id": dataset.attrs.get("dataset_id", ""),
            "source_path": source_path,
            "recipe_ids": [recipe.id for recipe in matched_plans],
        }
    )

matches

Inspect one selected recipe and its ordered fix ids.

In [ ]:
selected = store.get_recipe("cmip6_decadal.full")

{
    "id": selected.id,
    "description": selected.description,
    "match": selected.match.model_dump() if selected.match else None,
    "steps": [step.id for step in selected.steps],
}

In [ ]:
store.close()